# Modelo de prueba

En este notebook, se verá más en detalle la arquitectura transformer explorando el modelo exclusivo de decodificador (decoder-only) [model](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct) `microsoft/Phi-3-mini-4k-instruct`. Se realizará un texto de respuesta a partir de un prompt.

## 1. Configuración

Comenzamos configurando el laboratorio instalando las librerías requeridas (`transformers` y `accelerate`) e ignorando las advertencias (warnings). 

In [ ]:

#pip install transformers>=4.41.2 accelerate>=0.31.0
import warnings
warnings.filterwarnings('ignore')

## 2. Cargamos el LLM

Cargamos el modelo y su tokenizador. Para ello, primero importamos las clases: `AutoModelForCausalLM` y `AutoTokenizer`. Realizaremos dos métodos para procesar una oración, el primero consiste en aplicar primero el tokenizador y luego el modelo en dos pasos separados. El segundo es crear un objeto pipeline que envuelva los dos pasos y luego aplicar el pipeline a la oración. Por tanto, también importaremos la clase `pipeline`.


La librería transformers tiene dos tipos de clases de modelos: <code>AutoModelForCausalLM</code> y <code>AutoModelForMaskedLM</code>. Los modelos de lenguaje causales representan los modelos exclusivos de decodificador (decoder-only) que se utilizan para la generación de texto. Se describen como causales porque, para predecir el siguiente token, el modelo solo puede prestar atención a los tokens precedentes situados a la izquierda. Los modelos de lenguaje enmascarados (masked) representan los modelos exclusivos de codificador (encoder-only) que se utilizan para una representación rica de texto. Se describen como enmascarados porque están entrenados para predecir un token enmascarado u oculto en una secuencia.

In [2]:
#Importamos las clases necesarias
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [3]:
#Carga del modelo y tokenizador
tokenizer = AutoTokenizer.from_pretrained("../models/microsoft/Phi-3-mini-4k-instruct")

model = AutoModelForCausalLM.from_pretrained(
    "../models/microsoft/Phi-3-mini-4k-instruct",
    device_map="cpu",
    torch_dtype="auto",
    trust_remote_code=True,
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Ahora podemos envolver el modelo y el tokenizador en un objeto [pipeline](https://huggingface.co/docs/transformers/en/main_classes/pipelines#transformers.pipeline) que tenga "text-generation" (generación de texto) como identificador de tarea.

In [4]:
#Creacion de pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False, # False means to not include the prompt text in the returned text
    max_new_tokens=50, 
    do_sample=False, # no randomness in the generated text
)

## 3. Generación de una respuesta de texto a partir de un prompt

Ahora utilizaremos el objeto pipeline (nombrado como generator) para generar una respuesta que conste de 50 tokens para el prompt dado.

In [5]:
prompt = "Escribe un poema sobre la ciudad de Sevilla "

output = generator(prompt)

print(output[0]['generated_text'])

You are not running the flash-attention implementation, expect numerical differences.




Sevilla, ciudad de fuego y luz,
donde el río Guadalquivir fluye con pasión.

En su corazón, la catedral de Santa María,
con sus torres


## Exploración de la arquitectura del modelo

Ahora podemos imprimir por pantalla el modelo para echar un vistazo a su arquitectura.

In [6]:
model

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
          (rotary_emb): Phi3RotaryEmbedding()
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLU()
        )
        (input_layernorm): Phi3RMSNorm()
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
        (post_attention_layernorm): Phi3RMSNorm()
      )
    )
    (norm): Phi3RMSNorm()
  )
  (lm_head): Linear(in_features=3072, out_features=3206

El tamaño del vocabulario es de 32,064 tokens, y el tamaño del vector de incrustación (embedding) para cada token es de 3,072.

In [7]:
model.model.embed_tokens

Embedding(32064, 3072, padding_idx=32000)

Podemos únicamente imprimir la pila de bloques del transformer sin el componente de la cabecera del modelo de lenguaje (LM head).

In [8]:
model.model

Phi3Model(
  (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
  (embed_dropout): Dropout(p=0.0, inplace=False)
  (layers): ModuleList(
    (0-31): 32 x Phi3DecoderLayer(
      (self_attn): Phi3Attention(
        (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        (rotary_emb): Phi3RotaryEmbedding()
      )
      (mlp): Phi3MLP(
        (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
        (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
        (activation_fn): SiLU()
      )
      (input_layernorm): Phi3RMSNorm()
      (resid_attn_dropout): Dropout(p=0.0, inplace=False)
      (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      (post_attention_layernorm): Phi3RMSNorm()
    )
  )
  (norm): Phi3RMSNorm()
)

Hay 32 bloques o capas transformer. Puede accederse a cualquier bloque en particular.

In [9]:
model.model.layers[0]

Phi3DecoderLayer(
  (self_attn): Phi3Attention(
    (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
    (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (mlp): Phi3MLP(
    (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
    (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
    (activation_fn): SiLU()
  )
  (input_layernorm): Phi3RMSNorm()
  (resid_attn_dropout): Dropout(p=0.0, inplace=False)
  (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
  (post_attention_layernorm): Phi3RMSNorm()
)

## Generación de un solo token para un prompt

Anteriormente usamos el objeto Pipeline para generar una respuesta de texto completa a partir de un prompt. El pipeline proporciona una abstracción del proceso subyacente de generación de texto. En realidad, cada token en el texto se genera uno por uno.

Ahora procederemos a introducirle un prompt al modelo y verificaremos el primer token que genera.

In [10]:
prompt = "La persona que descubrió América es"

Primero necesitamos tokenizar el prompt y obtener los IDs de los tokens.

In [11]:
#Tokenizamos el prompt de entrada
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
input_ids

tensor([[  997, 21411,   712,  5153,   431, 24584, 19537,   831]])

Pasamos ahora los IDs de los tokens al bloque transformer (antes de la cabecera LM head).

In [12]:
#Obtenemos el output del modelo antes de lm_head
model_output = model.model(input_ids)

El bloque transformer genera para cada token un vector de tamaño 3,072 (tamaño del embedding). Mostramos la forma (shape) de este resultado.

In [13]:
#Obtenemos la forma del modelo antes de lm_head
model_output[0].shape

torch.Size([1, 8, 3072])

El primer número representa el tamaño del lote (batch size), que en este caso es 1 ya que tenemos un solo prompt. El segundo número, 5, representa la cantidad de tokens. Y finalmente, 3,072 representa el tamaño del embedding (el tamaño del vector que corresponde a cada token).

A continuación obtenemosla salida de la cabecera LM (LM head).

In [14]:
#Obtenemos la salida de lm_head
lm_head_output = model.lm_head(model_output[0])

In [15]:
lm_head_output.shape

torch.Size([1, 8, 32064])

La cabecera LM genera, para cada token en el prompt de entrada, un vector de tamaño 32,064 (tamaño del vocabulario). Así que hay 5 vectores, cada uno de tamaño 32,064. Cada vector se puede mapear a una distribución de probabilidad que muestra la probabilidad de que cada token del vocabulario aparezca después del token dado en el prompt de entrada.

Dado que estamos interesados en generar el token de salida que viene después del último token en el prompt de entrada ("is"), nos enfocaremos en el último vector. Por lo tanto, en la siguiente celda, `lm_head_output[0,-1]` es un vector de tamaño 32,064 a partir del cual puedes generar el token que viene después ("is"). Puedes hacerlo buscando el ID del token que corresponda al valor más alto en el vector `lm_head_output[0,-1]` (utilizando `argmax(-1)`, donde -1 significa a lo largo del último eje en este caso).

In [16]:
token_id = lm_head_output[0,-1].argmax(-1)
token_id

tensor(12546)

Finalmente, decodifiquemos el ID del token devuelto.

In [17]:
tokenizer.decode(token_id)

'Crist'

Dado que solo está generando el primer token y no la palabra completa, solo nos ha mostrado en este caso 'Crist' en lugar de mostrarnos todos los tokens dando lugar a la respuesta que esperábamos que diera: 'Cristóbal Colón'